## E5-base-v2 fine-tuning on SQuARE — generic contrastive, capacity-matched baseline

Fine-tunes **`intfloat/e5-base-v2`** (~110M params) on the **SQuARE** benchmark
with the same **generic in-batch-negatives InfoNCE** recipe used for the Qwen3
baseline, but on a backbone **matched in scale to SparqlCE's own text encoder**
(`microsoft/codebert-base`, ~125M) rather than a much larger LLM.

**Why this baseline exists.** A fine-tuned Qwen3-Embedding-0.6B beats SparqlCE
on RQ1, but Qwen3 is ~5x larger and far more heavily pretrained — so that gap
conflates *backbone scale* with *training design*. E5-base-v2 is roughly
CodeBERT's size, so fine-tuning it with the same generic recipe isolates the
design question: if SparqlCE beats a scale-matched, generically fine-tuned
encoder, the gap credits SparqlCE's **graph encoder + calibrated graded
objective**, not just "we used a bigger/pretrained model."

**Recipe (matched to SparqlCE's own schedule):**

    backbone   = intfloat/e5-base-v2 (encoder, mean-pooled, "query: " prefix)
    training   = FULL fine-tune (all params), matching SparqlCE's own full
                 fine-tune of CodeBERT — NOT LoRA
    loss       = in-batch-negatives InfoNCE, scale = 20 (== SparqlCE tau=0.05)
    lr         = 1e-5 (fixed, matching SparqlCE's own CodeBERT LR)
    epochs     = 20 (fixed, matching SparqlCE's own schedule; no early stopping)
    selection  = best val MRR (embedding ranking, same protocol as SparqlCE)
    splits     = paper's fixed split (splits_dbpedia.json)

SparqlCE-specific components (Jaccard calibration, graph encoder, augmentation
curriculum) are deliberately EXCLUDED — those are SparqlCE's contribution.
This is the DBPEDIA run; see the other notebook for the paired KG.

In [ ]:
# Dependencies (uncomment on a fresh environment)
# %pip install -q torch transformers>=4.51 accelerate tqdm


In [1]:
import os
import sys
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from tqdm.auto import tqdm

# ============================================================================
# 1. CONFIG — verify these paths match your repo before running
# ============================================================================
KG = "dbpedia"   # "dbpedia" | "wikidata"

PATHS = {
    "dbpedia": {
        "data":   "../../datasets/dbpedia-triplets-dataset.json",
        "splits": "../../datasets/splits_dbpedia.json",
    },
    "wikidata": {
        "data":   "../../datasets/wikidata-triplets-dataset.json",
        "splits": "../../datasets/splits_wikidata.json",
    },
}
DATA_PATH   = PATHS[KG]["data"]
SPLITS_PATH = PATHS[KG]["splits"]
CKPT_DIR    = f"../../checkpoints/e5_square_{KG}"

MODEL_NAME  = "intfloat/e5-base-v2"
E5_PREFIX   = "query: "        # E5 convention; applied to every SPARQL string,
                               # matching how zero-shot E5 is evaluated in
                               # evaluate_rq1.py (transform=lambda q: f"query: {q}")

# Generic (non-SparqlCE) contrastive fine-tuning recipe on a CAPACITY-MATCHED
# backbone (~110M, close to CodeBERT's ~125M). SparqlCE-specific components
# (Jaccard calibration, graph encoder, augmentation curriculum) are
# DELIBERATELY EXCLUDED so this baseline credits only a generic contrastive
# adaptation of a comparable-scale encoder on SQuARE (see BASELINE_REDESIGN.md).
SEED         = 42
WEIGHT_DECAY = 1e-2
GRAD_CLIP    = 1.0

# Epoch budget: fixed at 20, matching SparqlCE's own training schedule
# exactly (see SUPPLEMENTARY.md hyperparameter table) — no early stopping.
EPOCHS       = 20

# Training settings
BATCH_SIZE   = 16            # per-step; each anchor yields one pos + one neg pair
                             # -> 32 contrastive candidates/step, matching SparqlCE's
                             # own BATCH_SIZE=32 in-batch-negative pool size
GRAD_ACCUM   = 2             # effective batch 32, matches the SparqlCE recipe
SCALE        = 20.0          # InfoNCE temperature scale == 1 / SparqlCE's tau=0.05
MAX_LEN      = 512           # tokens per SPARQL query
WARMUP_FRAC  = 0.05
AUGMENT_RENAME_PROB = 0.0    # no domain-specific augmentation in the standard baseline

# Learning rate: fixed, matching SparqlCE's own CodeBERT fine-tuning LR (1e-5).
# E5-base-v2 is full-fine-tuned here (all params), exactly as SparqlCE fully
# fine-tunes its own similar-scale CodeBERT backbone — so the two backbones
# get identical treatment (full FT, same LR, same fixed schedule).
LR = 1e-5

/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================================
# 2. Setup: repo import path, seeds, device
# ============================================================================
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_ROOT = str(Path.cwd().parent.parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# No dependency on sparqlce.dataset: AUGMENT_RENAME_PROB is fixed at 0.0 for
# this baseline, so the SparqlCE-specific rename augmentation is never used
# (see Section 3 dataset class) and does not need to be imported.

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")


device = cuda


### 3. Data

Same SQuARE JSON schema as `SemanticSPARQLDataset`
(`anchor_query` / `positive_queries` / `negative_queries`), but the dataset
below is **text-only**: no graph builder, no PyG batching. Each item yields
an explicit `(anchor_text, positive_text, negative_text)` triple — one
positive pair and one dataset negative per anchor per step, used as the
labelled positive/negative in the in-batch InfoNCE loss (Section 6). Unlike
the SparqlCE recipe, the graded `answer_jaccard` values are not used
anywhere: this baseline only sees binary match / non-match membership, and
weights every negative (in-batch or dataset-provided) equally.


In [3]:
class SquareTextDataset(Dataset):
    """Anchor-level SQuARE dataset over raw SPARQL text.

    Text-side revision of ``sparqlce.dataset.SemanticSPARQLDataset``: same
    JSON parsing and augmentation, minus the graph construction and minus
    the graded Jaccard signal (only used here to pick match / non-match
    pairs used as the labelled positive/negative in the InfoNCE loss).
    """

    def __init__(self, datapath, augment_rename_prob=0.0, augment_seed=None):
        self.augment_rename_prob = float(augment_rename_prob)
        self._aug_rng = random.Random(augment_seed) if augment_seed is not None else random
        self.groups = []

        print(f"Loading dataset from {datapath}...")
        with open(datapath, "r", encoding="utf-8") as f:
            raw = json.load(f)

        for item in raw:
            anchor = item.get("anchor_query")
            if not anchor:
                continue
            pos_list = [
                p["query"] for p in item.get("positive_queries", [])
                if p.get("query") is not None
            ]
            neg_list = [
                n["query"] for n in item.get("negative_queries", [])
                if n.get("query") is not None
            ]
            if not pos_list or not neg_list:
                continue
            self.groups.append({"anchor": anchor, "positives": pos_list, "negatives": neg_list})

        print(f"Loaded {len(self.groups)} anchor groups.")

    def _maybe_augment(self, query):
        # No-op: this baseline never applies SparqlCE-specific augmentation
        # (rename curriculum). augment_rename_prob is accepted for interface
        # compatibility but intentionally unused.
        return query

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        g = self.groups[idx]
        pos_q = random.choice(g["positives"])
        neg_q = random.choice(g["negatives"])
        return (self._maybe_augment(g["anchor"]),
                self._maybe_augment(pos_q),
                self._maybe_augment(neg_q))


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)  # E5/BERT: default right padding

def _prefix(qs):
    """Prepend E5's "query: " instruction to every SPARQL string, matching
    how zero-shot E5 is evaluated in evaluate_rq1.py."""
    return [E5_PREFIX + q for q in qs]

def collate_pairs(batch):
    """Tokenize (anchor, positive, negative) triples into three padded batches."""
    anchors, positives, negatives = zip(*batch)
    a_tok = tokenizer(_prefix(list(anchors)),   padding=True, truncation=True,
                      max_length=MAX_LEN, return_tensors="pt")
    p_tok = tokenizer(_prefix(list(positives)), padding=True, truncation=True,
                      max_length=MAX_LEN, return_tensors="pt")
    n_tok = tokenizer(_prefix(list(negatives)), padding=True, truncation=True,
                      max_length=MAX_LEN, return_tensors="pt")
    return a_tok, p_tok, n_tok


In [4]:
# ============================================================================
# 3b. Fixed split (paper) with safe random fallback — unchanged from SparqlCE
# ============================================================================
dataset = SquareTextDataset(
    DATA_PATH,
    augment_rename_prob=AUGMENT_RENAME_PROB,
    augment_seed=SEED,
)
# Validation/test must see un-augmented queries: share the file, not the flags.
eval_dataset = SquareTextDataset(DATA_PATH, augment_rename_prob=0.0)

if os.path.exists(SPLITS_PATH):
    with open(SPLITS_PATH, "r") as f:
        splits = json.load(f)
    train_idx = splits.get("train_idx", splits.get("train"))
    val_idx   = splits.get("val_idx",   splits.get("valid_idx", splits.get("val")))
    test_idx  = splits.get("test_idx",  splits.get("test"))
    assert train_idx and val_idx and test_idx, f"Could not parse splits from {SPLITS_PATH}"

    train_dataset = Subset(dataset,      train_idx)
    val_dataset   = Subset(eval_dataset, val_idx)
    test_dataset  = Subset(eval_dataset, test_idx)
    print(f"Using FIXED split from {SPLITS_PATH}")
else:
    print(f"WARNING: fixed split file not found at {SPLITS_PATH}")
    print("WARNING: falling back to seeded random_split — numbers will NOT match the paper.")
    total   = len(dataset)
    n_train = int(0.80 * total)
    n_val   = int(0.10 * total)
    n_test  = total - n_train - n_val
    gen = torch.Generator().manual_seed(SEED)
    train_split, val_split, test_split = random_split(
        range(total), [n_train, n_val, n_test], generator=gen)
    train_dataset = Subset(dataset,      list(train_split.indices))
    val_dataset   = Subset(eval_dataset, list(val_split.indices))
    test_dataset  = Subset(eval_dataset, list(test_split.indices))

print(f"train={len(train_dataset)}  val={len(val_dataset)}  test={len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_pairs, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_pairs)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_pairs)


Loading dataset from ../../datasets/dbpedia-triplets-dataset.json...
Loaded 4713 anchor groups.
Loading dataset from ../../datasets/dbpedia-triplets-dataset.json...
Loaded 4713 anchor groups.
Using FIXED split from ../../datasets/splits_dbpedia.json
train=3770  val=471  test=472


### 4. Model: E5-base-v2 + FULL fine-tune

E5-base-v2 is a BERT-style bidirectional encoder: the query embedding is the
**mean of its token hidden states** (the pooling E5 was trained with),
L2-normalised — this replaces SparqlCE's GATv2 + mean-pool + projection head.
All parameters are trained (full fine-tune), matching SparqlCE's own full
fine-tune of CodeBERT; there is no LoRA and no auxiliary head.

In [5]:
def mean_pool(last_hidden_states, attention_mask):
    """Mean pooling over non-padding tokens, as E5 was trained with."""
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_states * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


class E5SparqlEmbedder(nn.Module):
    """E5-base-v2 backbone (full fine-tune), producing L2-normalised embeddings.

    Same output contract as ``HybridSPARQLEncoder.forward`` (unit-norm float
    embeddings), so the SparqlCE evaluation code applies verbatim. No auxiliary
    head: the same ``forward`` output is used for training (InfoNCE) and
    inference (Sections 9-10).
    """

    def __init__(self, model_name_or_path):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name_or_path)

    def forward(self, tok):
        out = self.backbone(input_ids=tok["input_ids"],
                            attention_mask=tok["attention_mask"])
        emb = mean_pool(out.last_hidden_state, tok["attention_mask"])
        return F.normalize(emb.float(), p=2, dim=1)


def build_model_and_optim(lr, epochs, steps_per_epoch):
    """Fresh full-fine-tune model + optimizer + scheduler at a given LR/epoch
    budget. All backbone parameters are trainable (no LoRA)."""
    m = E5SparqlEmbedder(MODEL_NAME).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    total_steps = max((steps_per_epoch // GRAD_ACCUM) * epochs, 1)
    sched = get_cosine_schedule_with_warmup(
        opt,
        num_warmup_steps=int(WARMUP_FRAC * total_steps),
        num_training_steps=total_steps,
    )
    return m, opt, sched


# One-off print of trainable-parameter count, with an explicit full-FT guard.
_probe, _, _ = build_model_and_optim(LR, 1, len(train_loader))
_total = sum(p.numel() for p in _probe.parameters())
_trainable = sum(p.numel() for p in _probe.parameters() if p.requires_grad)
print(f"total params:     {_total:,}")
print(f"trainable params: {_trainable:,}")
print(f"trainable %:      {100 * _trainable / _total:.2f}")
assert _trainable == _total, "Not a full fine-tune: some parameters are frozen."
del _probe
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1175.46it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


total params:     109,482,240
trainable params: 109,482,240
trainable %:      100.00


In [6]:
# ============================================================================
# 5. Loss — generic in-batch-negatives InfoNCE (NOT SparqlCE's loss).
#
# This is the standard contrastive recipe used by most open sentence-
# embedding fine-tunes (equivalent to sentence-transformers'
# MultipleNegativesRankingLoss with explicit hard negatives appended): for
# each anchor, its true positive is scored against every positive AND every
# dataset negative in the batch, softmax-normalised. No graded-similarity
# (Jaccard) regression term, no calibration, no hard-negative mining beyond
# the dataset's own annotated negatives, no graph encoder — those are
# SparqlCE's contribution and are deliberately excluded here
# (see BASELINE_REDESIGN.md).
# ============================================================================
def info_nce_loss(a_emb, p_emb, n_emb, scale):
    """In-batch-negatives InfoNCE: anchor i's positive is column i of
    concat(P, N); every other positive and every dataset negative in the
    batch acts as an additional, unweighted negative."""
    candidates = torch.cat([p_emb, n_emb], dim=0)          # [2B, d]
    logits = scale * (a_emb @ candidates.T)                # [B, 2B]
    labels = torch.arange(a_emb.size(0), device=a_emb.device)
    loss = F.cross_entropy(logits, labels)
    with torch.no_grad():
        acc = (logits.argmax(dim=1) == labels).float().mean()
    return loss, acc


In [7]:
# ============================================================================
# 6. Validation: rank each anchor's true positive against ALL positives in
# the split, using cosine similarity of the pooled embeddings only (no
# auxiliary head is involved — the same embeddings used for training).
# Identical protocol to SparqlCE — selects the embedding that retrieves
# correctly, not the one that collapses everything to a single point.
# ============================================================================
@torch.no_grad()
def evaluate_ranking(model, loader, device):
    model.eval()
    anchors, positives = [], []
    for a_tok, p_tok, _ in loader:
        a_tok = {k: v.to(device) for k, v in a_tok.items()}
        p_tok = {k: v.to(device) for k, v in p_tok.items()}
        anchors.append(model(a_tok).cpu())
        positives.append(model(p_tok).cpu())
    A = torch.cat(anchors)      # [N, d]
    P = torch.cat(positives)    # [N, d]
    sims  = A @ P.T             # [N, N]; diagonal is the true positive
    diag  = sims.diag().unsqueeze(1)
    ranks = (sims >= diag).sum(dim=1).float()   # 1-indexed rank of true positive
    return {
        "mrr":    (1.0 / ranks).mean().item(),
        "hits@1": (ranks == 1).float().mean().item(),
        "hits@5": (ranks <= 5).float().mean().item(),
    }


In [8]:
# ============================================================================
# 8. Training loop — generic in-batch-negatives InfoNCE fine-tuning at a
# fixed LR for a fixed 20 epochs (gradient accumulation), matching
# SparqlCE's own training schedule exactly. No early stopping: both sides
# train for the same fixed budget. Checkpoint selection is still by best
# val MRR across epochs (same as SparqlCE's own "Model selection: best
# validation MRR"), just without cutting the run short.
# ============================================================================
os.makedirs(CKPT_DIR, exist_ok=True)
best_mrr  = -1.0
best_path = os.path.join(CKPT_DIR, "best_model")
history   = []

model, optimizer, scheduler = build_model_and_optim(LR, EPOCHS, len(train_loader))
_n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {_n/1e6:.1f}M (full fine-tune)")

print(f"Training E5-base-v2 on SQuARE-{KG} for {EPOCHS} epochs "
      f"(lr={LR:.0e}, effective batch {BATCH_SIZE * GRAD_ACCUM}) "
      f"\u2014 fixed schedule, matching SparqlCE")
print("loss = InfoNCE(A, [P; N]) with in-batch negatives \u2014 "
      "generic contrastive, no Jaccard calibration, no graph")

for epoch in range(1, EPOCHS + 1):
    model.train()
    sum_loss, sum_acc, n_steps = 0.0, 0.0, 0
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS}")

    for step, (a_tok, p_tok, n_tok) in enumerate(pbar, start=1):
        a_tok = {k: v.to(device) for k, v in a_tok.items()}
        p_tok = {k: v.to(device) for k, v in p_tok.items()}
        n_tok = {k: v.to(device) for k, v in n_tok.items()}

        a_emb = model(a_tok)
        p_emb = model(p_tok)
        n_emb = model(n_tok)

        loss, acc = info_nce_loss(a_emb, p_emb, n_emb, SCALE)
        (loss / GRAD_ACCUM).backward()

        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                (p for p in model.parameters() if p.requires_grad), max_norm=GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        sum_loss += loss.item(); sum_acc += acc.item(); n_steps += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc.item():.4f}")

    val_metrics = evaluate_ranking(model, val_loader, device)
    history.append({"epoch": epoch,
                    "train_loss": sum_loss / n_steps,
                    "train_acc": sum_acc / n_steps,
                    **{f"val_{k}": v for k, v in val_metrics.items()}})
    print(f"Epoch {epoch:02d} | loss {sum_loss/n_steps:.4f} "
          f"acc {sum_acc/n_steps:.4f} | "
          f"val MRR {val_metrics['mrr']:.4f} "
          f"H@1 {val_metrics['hits@1']:.4f} H@5 {val_metrics['hits@5']:.4f}")

    # Selection by val MRR (embedding-space ranking), as in the SparqlCE
    # recipe. The full fine-tuned model is saved. No early
    # stopping \u2014 runs the full fixed 20 epochs regardless.
    if val_metrics["mrr"] > best_mrr:
        best_mrr = val_metrics["mrr"]
        model.backbone.save_pretrained(best_path)
        tokenizer.save_pretrained(best_path)
        print(f"  \u2713 new best val MRR {best_mrr:.4f} \u2014 model saved to {best_path}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 965.27it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable params: 109.5M (full fine-tune)
Training E5-base-v2 on SQuARE-dbpedia for 20 epochs (lr=1e-05, effective batch 32) — fixed schedule, matching SparqlCE
loss = InfoNCE(A, [P; N]) with in-batch negatives — generic contrastive, no Jaccard calibration, no graph


Epoch 01/20: 100%|██████████| 235/235 [02:33<00:00,  1.53it/s, acc=0.6875, loss=0.8543]


Epoch 01 | loss 1.5138 acc 0.5944 | val MRR 0.4323 H@1 0.3355 H@5 0.5308


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.76it/s]


...

  ✓ new best val MRR 0.5444 — model saved to ../../checkpoints/e5_square_dbpedia/best_model


Epoch 20/20: 100%|██████████| 235/235 [02:36<00:00,  1.50it/s, acc=0.9375, loss=0.1295]


Epoch 20 | loss 0.2748 acc 0.8907 | val MRR 0.5481 H@1 0.4140 H@5 0.6858


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.59it/s]

  ✓ new best val MRR 0.5481 — model saved to ../../checkpoints/e5_square_dbpedia/best_model


In [9]:
# ============================================================================
# 9. Sanity check on the training-file test split (pair-ranking metric)
#
# NOTE: this is only the quick diagonal-ranking check used during training.
# The numbers reported in the paper's RQ1 tables come from Section 10
# below, which follows the exact protocol of
# notebooks/evaluation-RQ1/evaluate_rq1.py.
# ============================================================================
print(f"Training complete. Best val MRR: {best_mrr:.4f} (lr={LR:.0e}, {EPOCHS} fixed epochs)")
print("Reloading best model and evaluating on the test split...")

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

best_model = E5SparqlEmbedder(best_path).to(device)
test_metrics = evaluate_ranking(best_model, test_loader, device)

print(f"\nSQuARE-{KG} test sanity check (fine-tuned E5-base-v2):")
for k, v in test_metrics.items():
    print(f"  {k:>7}: {v:.4f}")

with open(os.path.join(CKPT_DIR, "test_metrics.json"), "w") as f:
    json.dump({"kg": KG, "model": MODEL_NAME, "lr": LR, "epochs": EPOCHS, "best_val_mrr": best_mrr,
               "test": test_metrics, "history": history}, f, indent=2)
print(f"\nMetrics written to {os.path.join(CKPT_DIR, 'test_metrics.json')}")


Training complete. Best val MRR: 0.5481 (lr=1e-05, 20 fixed epochs)
Reloading best model and evaluating on the test split...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 903.28it/s, Materializing param=pooler.dense.weight]                               



SQuARE-dbpedia test sanity check (fine-tuned E5-base-v2):
      mrr: 0.5461
   hits@1: 0.4195
   hits@5: 0.6864

Metrics written to ../../checkpoints/e5_square_dbpedia/test_metrics.json


### 10. RQ1 evaluation on the held-out test set (paper tables)

This is the evaluation that fills the **"E5-base-v2 (fine-tuned)"** row in
the paper's RQ1 tables (`tables/tab-RQ1-DBpedia.tex` / `tab-RQ1-Wikidata.tex`).
It follows the **exact** protocol of `notebooks/evaluation-RQ1/evaluate_rq1.py`,
the same script used for all baselines and SparqlCE:

- **Data:** `datasets/{kg}_testset_RQ1.json` (DBpedia: 337 anchors, 1677 corpus
  queries) — *not* the training-file split used above for model selection.
- **Retrieval:** each anchor is ranked against the **global corpus** (all unique
  anchor/positive/negative queries in the test set), anchor self-match excluded.
- **Metrics:** MRR, Hits, nDCG (ranking) and P, R, F1 (set overlap) at
  k = 3, 5, 10, 20, averaged over anchors with ≥1 annotated positive.

The only model-specific part is the encoder: queries are embedded with the
fine-tuned E5-base-v2 model (best-val-MRR checkpoint, mean pooling,
"query: " prefix, L2-normalised), mirroring
`build_emb_map_sentence_transformer` in the script.


In [10]:
# ============================================================================
# 10a. Load the RQ1 test set and build corpus / eval items / relevance maps
#      — exactly as notebooks/evaluation-RQ1/{DBpedia,Wikidata}/06_eval_*.ipynb
# ============================================================================
_RQ1_DIR = str(Path(PROJECT_ROOT) / "notebooks" / "evaluation-RQ1")
if _RQ1_DIR not in sys.path:
    sys.path.insert(0, _RQ1_DIR)

from evaluate_rq1 import (  # noqa: E402
    load_triplets, build_global_corpus, build_eval_items,
    build_binary_relevance_maps, build_sorted_pools_global,
    avg_prf_at_k_annotated, avg_mrr_at_k_annotated, avg_hits_at_k_annotated,
    avg_ndcg_binary, report,
)

RQ1_TEST_PATH = f"../../datasets/{KG}_testset_RQ1.json"
TOPKS = [3, 5, 10, 20]   # same k values as the paper tables

rq1_data = load_triplets(RQ1_TEST_PATH)
rq1_corpus, rq1_seen_q = build_global_corpus(rq1_data)
rq1_eval_items = build_eval_items(rq1_data, rq1_corpus, rq1_seen_q)
rq1_relevance_maps = build_binary_relevance_maps(rq1_eval_items, rq1_corpus)

# For DBpedia this should print: anchors=337, corpus=1677 (as in the paper).
print(f"RQ1 test set loaded from {RQ1_TEST_PATH}")
print(f"Evaluation ready -> anchors: {len(rq1_eval_items)} | corpus: {len(rq1_corpus)}")


/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /local/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /local/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_cluster/_version_cuda.so
  import torch_geometric.typing
/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disa

RQ1 test set loaded from ../../datasets/dbpedia_testset_RQ1.json
Evaluation ready -> anchors: 462 | corpus: 1841


In [11]:
# ============================================================================
# 10b. Encode the RQ1 corpus with the fine-tuned model and rank
# ============================================================================
@torch.no_grad()
def build_emb_map_e5(model, queries, batch_size=16, max_length=MAX_LEN):
    """Encode raw SPARQL strings with the fine-tuned E5-base-v2 embedder.

    Uses the same tokenizer settings and mean pooling as training;
    the model already returns L2-normalised float embeddings, so cosine
    similarity is a plain dot product (as evaluate_rq1 assumes).
    """
    model.eval()
    emb_map = {}
    for i in tqdm(range(0, len(queries), batch_size), desc="Encoding RQ1 corpus"):
        batch = queries[i:i + batch_size]
        tok = tokenizer([E5_PREFIX + q for q in batch], padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt")
        tok = {k: v.to(device) for k, v in tok.items()}
        embs = model(tok).cpu().numpy()
        for q, e in zip(batch, embs):
            emb_map[q] = e
    return emb_map


# `best_model` holds the best-val-MRR model reloaded in Section 9. If you
# are re-running only this section in a fresh kernel, uncomment:
# best_model = E5SparqlEmbedder(best_path).to(device)

rq1_emb_map = build_emb_map_e5(best_model, rq1_corpus, batch_size=BATCH_SIZE)
rq1_sorted_pools = build_sorted_pools_global(rq1_emb_map, rq1_eval_items, rq1_corpus)

report(f"E5-base-v2 (fine-tuned) — RQ1 {KG}",
       rq1_sorted_pools, rq1_eval_items, rq1_relevance_maps, TOPKS)


Encoding RQ1 corpus: 100%|██████████| 116/116 [00:05<00:00, 20.02it/s]



E5-base-v2 (fine-tuned) — RQ1 dbpedia
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.3773  0.4503  0.3113  0.6497  0.7359    0.3773  0.5928
@5     0.3117  0.4997  0.2795  0.6637  0.7965    0.3117  0.6007
@10    0.2442  0.5794  0.2520  0.6716  0.8615    0.2442  0.6103
@20    0.1896  0.6811  0.2336  0.6749  0.9069    0.1896  0.6236


In [12]:
# ============================================================================
# 10c. Collect metrics per k, emit LaTeX rows for the paper tables, save JSON
# ============================================================================
rq1_metrics = {}
for k in TOPKS:
    p, r, f1 = avg_prf_at_k_annotated(rq1_sorted_pools, rq1_eval_items, k)
    rq1_metrics[k] = {
        "MRR":  avg_mrr_at_k_annotated(rq1_sorted_pools, rq1_eval_items, k),
        "Hits": avg_hits_at_k_annotated(rq1_sorted_pools, rq1_eval_items, k),
        "nDCG": avg_ndcg_binary(rq1_sorted_pools, rq1_relevance_maps, k),
        "P": p, "R": r, "F1": f1,
    }

# Rows in the exact column order of tables/tab-RQ1-<KG>.tex
# (per k: MRR, Hits, nDCG for the ranking subtable; P, R, F1 for set-overlap).
rank_row = " & ".join(f"{rq1_metrics[k][m]:.4f}"
                      for k in TOPKS for m in ("MRR", "Hits", "nDCG"))
ovlp_row = " & ".join(f"{rq1_metrics[k][m]:.4f}"
                      for k in TOPKS for m in ("P", "R", "F1"))

print("LaTeX row — ranking-metrics subtable:")
print(f"E5-base-v2 (fine-tuned) & {rank_row} \\\\")
print("\nLaTeX row — set-overlap subtable:")
print(f"E5-base-v2 (fine-tuned) & {ovlp_row} \\\\")

rq1_out_path = os.path.join(CKPT_DIR, f"rq1_test_metrics_{KG}.json")
with open(rq1_out_path, "w") as f:
    json.dump({
        "kg": KG,
        "model": f"{MODEL_NAME} (fine-tuned, full)",
        "protocol": "notebooks/evaluation-RQ1/evaluate_rq1.py",
        "testset": RQ1_TEST_PATH,
        "lr": LR,
        "epochs": EPOCHS,
        "anchors": len(rq1_eval_items),
        "corpus_size": len(rq1_corpus),
        "topks": TOPKS,
        "metrics": {str(k): rq1_metrics[k] for k in TOPKS},
    }, f, indent=2)
print(f"\nRQ1 metrics written to {rq1_out_path}")


LaTeX row — ranking-metrics subtable:
E5-base-v2 (fine-tuned) & 0.6497 & 0.7359 & 0.5928 & 0.6637 & 0.7965 & 0.6007 & 0.6716 & 0.8615 & 0.6103 & 0.6749 & 0.9069 & 0.6236 \\

LaTeX row — set-overlap subtable:
E5-base-v2 (fine-tuned) & 0.3773 & 0.4503 & 0.3113 & 0.3117 & 0.4997 & 0.2795 & 0.2442 & 0.5794 & 0.2520 & 0.1896 & 0.6811 & 0.2336 \\

RQ1 metrics written to ../../checkpoints/e5_square_dbpedia/rq1_test_metrics_dbpedia.json


### Notes

- **Paper numbers come from Section 10.** The RQ1 tables in the paper are
  computed on `datasets/{kg}_testset_RQ1.json` with the shared
  `evaluate_rq1.py` protocol; Section 9's pair-ranking numbers are only a
  training sanity check and are **not** comparable to the tables.
- **Comparability.** Fixed splits and the MRR-based model selection are
  identical to the SparqlCE training notebook; the in-batch negative pool
  size (32) and temperature/scale (τ=0.05 == scale=20) are matched too. The
  RQ1 evaluation reuses the exact same script as all baselines, so the
  Section 10 output slots directly into the paper's tables as the
  *fine-tuned* E5-base-v2 row (vs. the zero-shot E5-base-v2 baseline).
- **LR and epoch budget match SparqlCE's own fixed schedule directly.**
  `LR = 1e-5` and `EPOCHS = 20` (Section 1 CONFIG) are not swept or
  early-stopped — they're fixed to mirror SparqlCE's own non-swept
  training schedule exactly, so neither side gets extra tuning effort the
  other lacks.
- **Zero-shot reference.** The zero-shot E5-base-v2 row is already produced by
  `evaluate_rq1.py` (via `evaluate_sentence_transformer("intfloat/e5-base-v2",
  ...)`); this notebook adds the *fine-tuned* E5 row on the same corpus.
- **Sanity check before trusting Section 10 numbers.** A properly trained
  contrastive fine-tune should not underperform the zero-shot baseline on
  Hits/nDCG at any k — if it does, suspect a checkpoint-selection or
  pooling mismatch before reporting the numbers.
- **Wikidata. Set `KG = "wikidata"` in the CONFIG cell; the notebook expects
  `datasets/wikidata-triplets-dataset.json`, `datasets/splits_wikidata.json`,
  and `datasets/wikidata_testset_RQ1_matched.json` for Section 10.**
- **Memory / compute.** ~110M-param E5-base-v2, full fine-tune (fp32, no LoRA),
  fits comfortably on a single 16-24 GB GPU at batch 16 / 512 tokens. Reduce
  `BATCH_SIZE` and raise `GRAD_ACCUM` proportionally if you hit OOM — in-batch
  negatives mean the *per-step* batch size also sets how many negatives each
  anchor is contrasted against.
